# Напишите заголовок проекта здесь

- Автор:Туровская Мария 
- Дата:16.06.2026

### Цели и задачи проекта

<font color='#777778'>Сделать обзор игровых платформ, изучить объёмы продаж игр разных жанров и региональные предпочтения игроков. для статьи о развитии индустрии игр в начале XXI века.</font>

### Описание данных

<font color='#777778'>
Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. -Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.</font>

### Содержимое проекта



<font color='#777778'>

- Проверить корректность данных
- Провести предобработку данных
- Категоризировать игры по оценкам пользователей и экспертов 
- Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период
</font></font>

---

## 1. Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [1]:
import pandas as pd 

In [2]:
#Выгрузка данных из CSV файла
data = pd.read_csv('/datasets/new_games.csv')

In [3]:
#Выводим первые 5 строк данных
data.head(5)

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


- Познакомьтесь с данными: выведите первые строки и результат метода `info()`.


In [4]:
#Выводим информацию о данных
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датасет содержит 16956 строк и 11 столбцов. 
Изучим типы данных и их корректность:

- **Числовые значения с плавающей запятой (float64).** 4 столбца в одном из них год выпуска, что может быть неправильно 
- **Строковые данные (str).** 7 столбцов имеют тип данных `str`что в целом правильно, хотя можно изменить данные о продажах на float или integer 
   

После анализа типов данных видно, что большинство столбцов корректно представлены. Однако для оптимизации можно использовать целочисленные типы с уменьшенной разрядностью.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

- Выведите на экран названия всех столбцов датафрейма и проверьте их стиль написания.
- Приведите все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [5]:
#Выводим название всех столбцов
data.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [6]:
#Приводим к стилю snake_case названия всех столбцов и убираем пробелы
data.columns = data.columns.str.lower().str.replace(' ', '_')

In [7]:
#Проверим количество строк и столбцов в данных
data.shape

(16956, 11)

### 2.2. Типы данных

- Если встречаются некорректные типы данных, предположите их причины.
- При необходимости проведите преобразование типов данных. Помните, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. Сначала вам понадобится обработать пропуски, а затем преобразовать типы данных.

In [8]:
#Выводим типы данных в каждом столбце
data.dtypes 

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object


На стадии знакомства с данными было установлено, что большинство столбцов корректно представлены. 
Однако для оптимизации нужно перевести некоторые численные столбцы string к числовому типу данных, заменив строковые значения на пропуски, чтобы можно было эффективно анализировать продажи по регионам 
 

In [9]:
#Находим столбцы user_score,eu_sales,jp_sales и приведем к числовому типу данных и заменим строковые значения на пропуски
data['user_score'] = pd.to_numeric(data['user_score'], errors='coerce')
data['critic_score'] = pd.to_numeric(data['critic_score'], errors='coerce')
data['eu_sales'] = pd.to_numeric(data['eu_sales'], errors='coerce')
data['jp_sales'] = pd.to_numeric(data['jp_sales'], errors='coerce')


In [10]:
#Проверяем типы данных в каждом столбце после преобразования
data.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

### 2.3. Наличие пропусков в данных

- Посчитайте количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [11]:
#Выведем количество пропусков в каждом столбце в абсолютных числах и в процентах
missing_values_count = data.isna().sum()
missing_values_percentage = round((missing_values_count / len(data)) * 100, 2)
print("Количество пропусков в каждом столбце:")
print(missing_values_count)
print("\nПроцент пропусков в каждом столбце:")
print(missing_values_percentage)

Количество пропусков в каждом столбце:
name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

Процент пропусков в каждом столбце:
name                0.01
platform            0.00
year_of_release     1.62
genre               0.01
na_sales            0.00
eu_sales            0.04
jp_sales            0.02
other_sales         0.00
critic_score       51.39
user_score         54.66
rating             40.52
dtype: float64


- Изучите данные с пропущенными значениями. Напишите промежуточный вывод: для каких столбцов характерны пропуски и сколько их. Предположите, почему пропуски могли возникнуть. Укажите, какие действия с этими данными можно сделать и почему.


В данных наблюдаются пропуски в следующих столбцах:
- `Name`: в 2 строках (0.01% данных) отсутствует информация о названии игры. Это может затруднить идентификацию игр, если необходимо провести анализ по конкретным играм.
- `year of release`: в 275 строках (1.62% данных) отсутствует информация о годе выпуска на платформе. Пропуски в этом столбце могут повлиять на анализ по хронологическим признакам.
- `genre`: в 2 строках (0.01% данных) отсутствуют данные о жанре игры. Возможно, пропуски в этом поле связаны с пропусками в `Name`, учитывая общее количество пропущенных строк.
- `critics score`: в 8714 строках (51.4% данных) отсутствует информация об оценке критиков, что может повлиять на анализ восприятия игры пользователями и критиками. Чтобы у игры появился critic_score, нужно, чтобы на неё написали рецензии как минимум несколько авторитетных изданий (обычно от 4-7 сайтов). На тысячи мелких, бюджетных, региональных (например, чисто японских визуальных новелл) или инди-игр критики просто не обращают внимания. 
- `user score`: в 9268 строках (54.7% данных) отсутствует информация об оценке пользователей, что может повлиять на анализ восприятия игры и рейтинга.Если игру купили всего несколько сотен раз, у неё не наберется критическая масса оценок от игроков, чтобы сформировать user_score. Страница игры на условном Metacritic будет пустой.Если в датасете есть игры для ранних мобильных телефонов или специфических платформ, их оценки практически никогда не агрегировались. 
- `rating`: в 6871 строках (40.5% данных) отсутствует информация о рейтинге, что может повлиять на анализ рейтинга. Если игра была разработана и выпущена только в Японии (а таких игр на консолях Nintendo и PlayStation исторически колоссальное количество) или продавалась исключительно в Европе, издателям просто не нужно было платить деньги американской ESRB и проходить их сертификацию. В датасете у таких игр продажи в Америке (na_sales) будут равны нулю, а рейтинг ESRB — пропуском (NaN). Получение рейтинга ESRB — это платная процедура, которая требует от издателя времени и заполнения кучи документов. Мелкие инди-разработчики, которые выпускают игры только в цифровом виде (например, в Steam или ранних цифровых магазинах), часто игнорировали получение рейтинга ESRB ради экономии бюджета, так как цифровые площадки в прошлые годы не всегда жестко требовали этот рейтинг. 

- Обработайте пропущенные значения. Для каждого случая вы можете выбрать оптимальный, на ваш взгляд, вариант: заменить на определённое значение, оставить как есть или удалить.
- Если вы решите заменить пропуски на значение-индикатор, то убедитесь, что предложенное значение не может быть использовано в данных.
- Если вы нашли пропуски в данных с количеством проданных копий игры в том или ином регионе, их можно заменить на среднее значение в зависимости от названия платформы и года выхода игры.

- rating - Это текстовые (категориальные) данные. Для них невозможно посчитать медиану. Как мы выяснили раньше, 40% пропусков здесь из-за того, что игры продавались в Японии или Европе, где ESRB просто не работает. Оставив 'unknown', мы сможем исследовать эти игры как отдельную важную группу

- year_of_release (Год выпуска)- заполним медианой по платформе: пропусков по годам обычно мало (1-2%). Можно заполнить их медианным годом для конкретной платформы (ведь если игра вышла на PS3, то год её выпуска явно находится где-то между 2006 и 2014 годами) и переведем в целочисленный тип 
- name (название) и genre (жанр)- такие строки лучше просто удалить

- critic score, user score оставляем пропуски тк значений слишком много и заполнение медианой может исказить данные

In [12]:
# 1. Заполняем пропуски
data['year_of_release'] = data['year_of_release'].fillna(data.groupby('platform')['year_of_release'].transform('median'))

# 2. Округляем и переводим в тип int
data['year_of_release'] = data['year_of_release'].round().astype('int')

In [13]:
#Удалим пропуски в столбцах name и genre
data = data.dropna(subset=['name', 'genre'])

### 2.4. Явные и неявные дубликаты в данных

- Изучите уникальные значения в категориальных данных, например с названиями жанра игры, платформы, рейтинга и года выпуска. Проверьте, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- При необходимости проведите нормализацию данных с текстовыми значениями. Названия или жанры игр можно привести к нижнему регистру, а названия рейтинга — к верхнему.

In [14]:
#Выведем уникальные значения в столбце genre, platform, name, rating, year_of_release
print("Уникальные значения в столбце genre:")
print(data['genre'].unique())
print("Уникальные значения в столбце name:")
print(data['name'].unique())
print("Уникальные значения в столбце rating:")
print(data['rating'].unique())


Уникальные значения в столбце genre:
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
Уникальные значения в столбце name:
['Wii Sports' 'Super Mario Bros.' 'Mario Kart Wii' ...
 'Woody Woodpecker in Crazy Castle 5' 'LMA Manager 2007'
 'Haitaka no Psychedelica']
Уникальные значения в столбце rating:
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


In [15]:
#Переводим значения в столбце genre, name в нижний регистр, а rating в верхний регистр 
data['genre'] = data['genre'].str.lower()
data['name'] = data['name'].str.lower()
data['rating'] = data['rating'].str.upper()


In [16]:
#Выведем уникальные значение в столбце genre, name, rating после преобразования
print("Уникальные значения в столбце genre после преобразования:")
print(data['genre'].unique())
print("Уникальные значения в столбце name после преобразования:")
print(data['name'].unique())
print("Уникальные значения в столбце rating после преобразования:")
print(data['rating'].unique())


Уникальные значения в столбце genre после преобразования:
['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']
Уникальные значения в столбце name после преобразования:
['wii sports' 'super mario bros.' 'mario kart wii' ...
 'woody woodpecker in crazy castle 5' 'lma manager 2007'
 'haitaka no psychedelica']
Уникальные значения в столбце rating после преобразования:
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


- После того как нормализуете данные и устраните неявные дубликаты, проверьте наличие явных дубликатов в данных.

- Напишите промежуточный вывод: укажите количество найденных дубликатов и действия по их обработке.

In [17]:
#Проверяем наличие явных дубликатов в данных
duplicates_count = data.duplicated().sum()
print(f"Количество явных дубликатов в данных: {duplicates_count}")

Количество явных дубликатов в данных: 241


In [18]:
#Удаляем дубликаты одной строчкой, чтобы они не раздували статистику, и сбрасываем индекс таблицы:
data = data.drop_duplicates()
data = data.reset_index(drop=True)

In [19]:
#Проверяем наличие дубликатов в данных после удаления
duplicates_count_after_removal = data.duplicated().sum()
print(f"Количество явных дубликатов в данных после удаления: {duplicates_count_after_removal}")

Количество явных дубликатов в данных после удаления: 0


- В процессе подготовки данных вы могли что-либо удалять, например строки с пропусками или ошибками, дубликаты и прочее. В этом случае посчитайте количество удалённых строк в абсолютном и относительном значениях.

In [20]:
#Выведем количество удаленных строк в абсолятных числах и в процентах
rows_removed = duplicates_count - duplicates_count_after_removal
percentage_removed = (rows_removed / len(data)) * 100
print(f"Количество удаленных строк: {rows_removed}")
print(f"Процент удаленных строк: {percentage_removed:.2f}%")

Количество удаленных строк: 241
Процент удаленных строк: 1.44%


- После проведения предобработки данных напишите общий промежуточный вывод.

Были удалены пропуски в столбцах name и genre, а также дубликаты строк. 
Столбцы genre и name были приведены к нижнему регистру, а столбец rating - к верхнему регистру.


---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.

In [21]:
#Создаем новый датафрейм df_actual по периоду с 20000 года по 2013 включительно 
df_actual = data[(data['year_of_release'] >= 2000) & (data['year_of_release'] <= 2013)].copy()

---

## 4. Категоризация данных
    
Проведите категоризацию данных:
- Разделите все игры по оценкам пользователей и выделите такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [22]:
#Разделим все игры по оценкам пользователей и выделим категории: 
# высокая оценка (от 8 до 10 включительно)
# средняя оценка (от 3 до 8, не включая правую границу интервала)
# низкая оценка (от 0 до 3, не включая правую границу интервала)
def categorize_user_score(score):
    if 8 <= score <= 10:
        return 'high'
    elif 3 <= score < 8:
        return 'medium'
    elif 0 <= score < 3:
        return 'low'
    else:
        return 'unknown'


In [23]:
#Выведем количество игр с оценкой пользователя high, medium, low и unknown
df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)
user_score_counts = df_actual['user_score_category'].value_counts()
print(user_score_counts)


unknown    6415
medium     4159
high       2328
low         119
Name: user_score_category, dtype: int64


- Разделите все игры по оценкам критиков и выделите такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [24]:
#Разделим все игры по оценкам критиков и выделим категории: 
# высокая оценка (от 80 до 100 включительно) 
# средняя оценка (от 30 до 80, не включая правую границу интервала) 
# и низкая оценка (от 0 до 30, не включая правую границу интервала)
def categorize_critic_score(score):
    if score >= 80 and score <= 100:
        return 'high'
    elif score >= 30 and score < 80:
        return 'medium'
    elif score >= 0 and score < 30:
        return 'low'
    else:
        return 'unknown'

In [25]:
#Выведем количество игр с оценкой критиков high, medium, low и unknown
df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)
critic_score_category_counts = df_actual['critic_score_category'].value_counts()
print(critic_score_category_counts)

unknown    5703
medium     5536
high       1724
low          58
Name: critic_score_category, dtype: int64


- После категоризации данных проверьте результат: сгруппируйте данные по выделенным категориям и посчитайте количество игр в каждой категории.

In [26]:
#Сгруппируем данные по категориям и посчитаем количество игр в каждой категории
user_score_category_counts = df_actual['user_score_category'].value_counts()
critic_score_category_counts = df_actual['critic_score_category'].value_counts()
print("Количество игр в каждой категории по оценкам пользователей:")
print(user_score_category_counts)
print("\nКоличество игр в каждой категории по оценкам критиков:")
print(critic_score_category_counts) 

Количество игр в каждой категории по оценкам пользователей:
unknown    6415
medium     4159
high       2328
low         119
Name: user_score_category, dtype: int64

Количество игр в каждой категории по оценкам критиков:
unknown    5703
medium     5536
high       1724
low          58
Name: critic_score_category, dtype: int64


- Выделите топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [27]:
# Выведем названия топ-7 платформ по количеству игр выпущенных за весь актуальных период с индексами
# количества выпущенных игр 
top_platform_counts = df_actual['platform'].value_counts().head(7).reset_index()
top_platform_counts.columns = ['platform', 'count']
top_platform_counts.index = top_platform_counts.index + 1
print(top_platform_counts)


  platform  count
1      PS2   2161
2       DS   2150
3      Wii   1309
4      PSP   1196
5     X360   1151
6      PS3   1112
7       XB    824


In [28]:
#Выведем топ 10 игр по количеству проданных копий в Северной Америке, чтобы определить, какие игры были наиболее популярными и успешными на этом рынке.
top_na_sales_games = df_actual.groupby('name')['na_sales'].sum().sort_values(ascending=False).head(10)
print("Топ 10 игр по количеству проданных копий в Северной Америке, млн шт:")
print(top_na_sales_games)

Топ 10 игр по количеству проданных копий в Северной Америке, млн шт:
name
wii sports                        41.36
call of duty: black ops           17.57
grand theft auto v                16.68
mario kart wii                    15.68
wii sports resort                 15.61
call of duty: modern warfare 3    15.54
kinect adventures!                15.00
call of duty: ghosts              14.94
new super mario bros. wii         14.44
call of duty: black ops ii        14.08
Name: na_sales, dtype: float64


In [29]:
#Выведем топ 5 продаваемых жанров в Северной Америке, чтобы определить, какие жанры видеоигр были наиболее популярными и успешными на этом рынке.
top_na_sales_genres = df_actual.groupby('genre')['na_sales'].sum().sort_values(ascending=False).head(5)
print("Топ 5 продаваемых жанров в Северной Америке:")
print(top_na_sales_genres)

Топ 5 продаваемых жанров в Северной Америке:
genre
action     691.98
sports     555.06
shooter    420.35
misc       357.92
racing     265.33
Name: na_sales, dtype: float64


---

## 5. Итоговый вывод

В конце напишите основной вывод и отразите, какую работу проделали. Не забудьте указать описание среза данных и новых полей, которые добавили в исходный датасет.

Топ платформа по выпуску игр- PS2. Топ жанр по продажам в Северной Америке- action. Был создан новый датафрейм для актуализации данных по периоду с 2000 по 2013 годы. В исходный датасет новых данных не добавлялось. 